# Qwen2.5 (small), from scratch

Goal: implement the architectural pieces that differ between GPT-2 and Qwen2.5, individually, then assemble them into a full small Qwen2 model.

Four differences from `gpt2_small.ipynb` to build one at a time: **RoPE** (rotary positional embeddings, replacing learned `wpe`), **RMSNorm** (replacing `LayerNorm`), **grouped-query attention / GQA** (fewer KV heads than query heads, replacing standard multi-head attention), **SwiGLU** (replacing the GELU-based MLP).

Build order: GQA -> RoPE -> RMSNorm -> SwiGLU (each tested standalone against GPT-2's equivalent piece) -> assemble into the full `Qwen2` model -> (optionally) load real pretrained weights and verify.

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- TODO: GroupedQueryAttention(n_embd, n_head, n_kv_heads) ---
# covering MHA (n_kv_heads == n_head), GQA (1 < n_kv_heads < n_head), MQA (n_kv_heads == 1)


# --- MHA (copied from gpt2_small.ipynb) ---
class CausalSelfAttentionMHA(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.d = n_embd
        self.nh = n_head
        self.dk = self.d// self.nh
        self.c_attn = nn.Linear(self.d, 3 * self.d)
        self.c_proj = nn.Linear(self.d, self.d)
        
    def forward(self, x):
        # x: [B, T, d]
        B, T, _ = x.shape

        ### projection
        x = self.c_attn(x) # [B, T, 3 * d]
        Q, K, V = x.split(self.d, dim=-1) 

        ### reshape
        Q = Q.view(B, T, self.nh, self.dk).transpose(1, 2)
        K = K.view(B, T, self.nh, self.dk).transpose(1, 2)
        V = V.view(B, T, self.nh, self.dk).transpose(1, 2) # [B, n, T, dk]

        ### scaled dot product
        attention_z = Q @ K.transpose(-2, -1)/ (self.dk ** .5) # [B, n, T, T]
        mask = torch.tril(torch.ones(T, T)).bool() 
        attention_mask = attention_z.masked_fill(~mask, float("-inf"))
        attention_scores = torch.softmax(attention_mask, dim=-1)
        out = attention_scores @ V  #   [B, n, T, d] 
        # out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    
        ### merging
        out = out.transpose(1, 2).contiguous().reshape(B, T, -1) # [B, T, d]
        out = self.c_proj(out) #[B, T, d]

        return out


x = torch.randn([2, 10, 16])
att = CausalSelfAttentionMHA(16, 2)
y = att(x)
print(x.shape, y.shape)


# --- MQA (n_kv_heads == 1) ----
class CausalSelfAttentionMQA(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.d = n_embd
        self.nh = n_head
        self.nhkv = 1
        self.dk = self.d// self.nh
        self.k_proj = nn.Linear(self.d, self.dk)
        self.v_proj = nn.Linear(self.d, self.dk)
        self.q_proj = nn.Linear(self.d, self.d)
        self.c_proj = nn.Linear(self.d, self.d)
        
    def forward(self, x):
        # x: [B, T, d]
        B, T, _ = x.shape

        ### projection
        Q = self.q_proj(x) # [B, T, d]
        K = self.k_proj(x) # [B, T, dk]
        V = self.v_proj(x) # [B, T, dk]
        
        ### reshape
        Q = Q.view(B, T, self.nh, self.dk).transpose(1, 2) # [B, n, T, dk]
        K = K.view(B, T, self.nhkv, self.dk).transpose(1, 2)# [B, 1, T, dk]
        V = V.view(B, T, self.nhkv, self.dk).transpose(1, 2) # [B, 1, T, dk]

        ### scaled dot product
        attention_z = Q @ K.transpose(-2, -1)/ (self.dk ** .5) # [B, n, T, T]
        mask = torch.tril(torch.ones(T, T)).bool() 
        attention_mask = attention_z.masked_fill(~mask, float("-inf"))
        attention_scores = torch.softmax(attention_mask, dim=-1)
        out = attention_scores @ V  #   [B, n, T, d] 
        # out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    
        ### merging
        out = out.transpose(1, 2).contiguous().reshape(B, T, -1) # [B, T, d]
        out = self.c_proj(out) #[B, T, d]

        return out

x = torch.randn([2, 10, 16])
att = CausalSelfAttentionMQA(16, 2)
y = att(x)
print(x.shape, y.shape)

# --- GQA (n_kv_heads == nhkv) ----
class CausalSelfAttentionGQA(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_heads):
        super().__init__()
        assert n_head % n_kv_heads == 0, "wrong n_kv_heads value!"
        self.d = n_embd
        self.nh = n_head
        self.nhkv = n_kv_heads
        self.dk = self.d// self.nh
        self.k_proj = nn.Linear(self.d, self.nhkv * self.dk)
        self.v_proj = nn.Linear(self.d, self.nhkv * self.dk)
        self.q_proj = nn.Linear(self.d, self.d)
        self.c_proj = nn.Linear(self.d, self.d)
        
    def forward(self, x):
        # x: [B, T, d]
        B, T, _ = x.shape

        ### projection
        Q = self.q_proj(x) # [B, T, d]
        K = self.k_proj(x) # [B, T, dk]
        V = self.v_proj(x) # [B, T, dk]
        
        ### reshape
        Q = Q.view(B, T, self.nh, self.dk).transpose(1, 2) # [B, n, T, dk]
        K = K.view(B, T, self.nhkv, self.dk).transpose(1, 2)# [B, nk, T, dk]
        V = V.view(B, T, self.nhkv, self.dk).transpose(1, 2) # [B, nk, T, dk]

        ### scaled dot product
        K = K.repeat_interleave(self.nh//self.nhkv, dim=1)
        attention_z = Q @ K.transpose(-2, -1)/ (self.dk ** .5) # [B, n, T, T]
        mask = torch.tril(torch.ones(T, T)).bool() 
        attention_mask = attention_z.masked_fill(~mask, float("-inf"))
        attention_scores = torch.softmax(attention_mask, dim=-1)
        V = V.repeat_interleave(self.nh//self.nhkv, dim=1)
        out = attention_scores @ V  #   [B, n, T, d] 
        # out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    
        ### merging
        out = out.transpose(1, 2).contiguous().reshape(B, T, -1) # [B, T, d]
        out = self.c_proj(out) #[B, T, d]

        return out

x = torch.randn([2, 10, 16])
att = CausalSelfAttentionGQA(16, 4, 2)
y = att(x)
print(x.shape, y.shape)


torch.Size([2, 10, 16]) torch.Size([2, 10, 16])
torch.Size([2, 10, 16]) torch.Size([2, 10, 16])
torch.Size([2, 10, 16]) torch.Size([2, 10, 16])
